# Clustering Lab

 
Based of the amazing work you did in the Movie Industry you've been recruited to the NBA! You are working as the VP of Analytics that helps support a head scout, Mr. Rooney, for the worst team in the NBA probably the Wizards. Mr. Rooney just heard about Data Science and thinks it can solve all the team's problems!!! He wants you to figure out a way to find players that are high performing but maybe not highly paid that you can steal to get the team to the playoffs! 

In this document you will work through a similar process that we did in class with the NBA data (NBA_Perf_22 and nba_salaries_22), merging them together. This is from 22-23 season, feel free to update to 2023-24 season if you want.
# Data Sources:

https://www.basketball-reference.com/leagues/NBA_2024_totals.html # reference for performance data
https://www.basketball-reference.com/contracts/players.html # reference for salary data


Details: 

- Determine a way to use clustering to estimate based on performance if 
players are under or over paid, generally. 

- Then select players you believe would be best for your team and explain why. Do so in three categories: 
    * Examples that are not good choices (3 or 4) 
    * Several options that are good choices (3 or 4)
    * Several options that could work, assuming you can't get the players in the good category (3 or 4)

- You will decide the cutoffs for each category, so you should be able to explain why you chose them.

- Provide a well commented and clean report of your findings in a separate notebook that can be presented to Mr. Rooney, keeping in mind he doesn't understand...anything. Include a rationale for variables you included in the model, details on your approach and a overview of the results with supporting visualizations. 


Hints:

- Salary is the variable you are trying to understand 
- When interpreting you might want to use graphs that include variables that are the most correlated with Salary
- You'll need to scale the variables before performing the clustering
- Be specific about why you selected the players that you did, more detail is better
- Use good coding practices, comment heavily, indent, don't use for loops unless totally necessary and create modular sections that align with some outcome. If necessary create more than one script,list/load libraries at the top and don't include libraries that aren't used. 
- Be careful for non-traditional characters in the players names, certain graphs won't work when these characters are included.


In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

In [2]:
# Reading in the data
perf_data = pd.read_csv("../data/NBA_Perf_22.csv", encoding='latin1')
sal_data = pd.read_csv("../data/nba_salaries_22.csv")

In [3]:
perf_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 812 entries, 0 to 811
Data columns (total 29 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Player  812 non-null    object 
 1   Pos     812 non-null    object 
 2   Age     812 non-null    int64  
 3   Tm      812 non-null    object 
 4   G       812 non-null    int64  
 5   GS      812 non-null    int64  
 6   MP      812 non-null    float64
 7   FG      812 non-null    float64
 8   FGA     812 non-null    float64
 9   FG%     797 non-null    float64
 10  3P      812 non-null    float64
 11  3PA     812 non-null    float64
 12  3P%     740 non-null    float64
 13  2P      812 non-null    float64
 14  2PA     812 non-null    float64
 15  2P%     784 non-null    float64
 16  eFG%    797 non-null    float64
 17  FT      812 non-null    float64
 18  FTA     812 non-null    float64
 19  FT%     715 non-null    float64
 20  ORB     812 non-null    float64
 21  DRB     812 non-null    float64
 22  TR

In [4]:
merged_data = pd.merge(perf_data, sal_data, on='Player', how='inner')
merged_data.head()

,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,FG%,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Salary
0,Precious Achiuwa,C,22,TOR,73,28,23.6,3.6,8.3,0.439,...,2.0,4.5,6.5,1.1,0.5,0.6,1.2,2.1,9.1,"$2,840,160"
1,Steven Adams,C,28,MEM,76,75,26.3,2.8,5.1,0.547,...,4.6,5.4,10.0,3.4,0.9,0.8,1.5,2.0,6.9,"$17,926,829"
2,Bam Adebayo,C,24,MIA,56,56,32.6,7.3,13.0,0.557,...,2.4,7.6,10.1,3.4,1.4,0.8,2.6,3.1,19.1,"$30,351,780"
3,Santi Aldama,PF,21,MEM,32,0,11.3,1.7,4.1,0.402,...,1.0,1.7,2.7,0.7,0.2,0.3,0.5,1.1,4.1,"$2,094,120"
4,Nickeil Alexander-Walker,SG,23,TOT,65,21,22.6,3.9,10.5,0.372,...,0.6,2.3,2.9,2.4,0.7,0.4,1.4,1.6,10.6,"$5,009,633"


In [5]:
# Dropping our duplicates
nba_data = merged_data.drop_duplicates()
nba_data = nba_data.drop('Tm', axis = 1)
nba_data = nba_data.dropna()

In [6]:
# Scaling the data
numeric_vars = nba_data.select_dtypes(include=['float64', 'int64'])
scaler = MinMaxScaler()
scaled_numeric_vars = scaler.fit_transform(numeric_vars)
scaled_df = pd.DataFrame(scaled_numeric_vars, columns=numeric_vars.columns, index=numeric_vars.index)
nba_data[numeric_vars.columns] = scaled_df

In [7]:
# Remove currency symbols and commas, then convert to float
nba_data['Salary'] = nba_data['Salary'].replace('$', '', regex=True)
nba_data['Salary'] = nba_data['Salary'].replace(',', '', regex=True)
nba_data['Salary'] = pd.to_numeric(nba_data['Salary'], errors='coerce')


In [8]:
#Run the clustering algo with your best guess for K
x = nba_data.iloc[:, 2:-1]
y = nba_data['Salary']

kmeans_nba = KMeans(n_clusters=3, random_state=67).fit(x)

In [9]:
#View the results
print(kmeans_nba.cluster_centers_) 
print(kmeans_nba.labels_)
print(kmeans_nba.inertia_)

[[0.29461898 0.53749092 0.1142396  0.41532563 0.18970588 0.22738361
  0.47452535 0.17696078 0.1979638  0.32092647 0.16946965 0.16491306
  0.3704951  0.60583571 0.0879257  0.0897249  0.62906782 0.17774936
  0.19210418 0.19379885 0.13180828 0.26019385 0.13130252 0.16796875
  0.29966987 0.19266693]
 [0.35830966 0.72415123 0.59441692 0.80060268 0.47606534 0.57342269
  0.44815473 0.48802083 0.50888088 0.36770313 0.35505319 0.36723066
  0.33666667 0.61316465 0.24161184 0.23343211 0.72494612 0.15217391
  0.30242407 0.26835664 0.36552373 0.453125   0.140625   0.38460286
  0.41326531 0.50226985]
 [0.31658692 0.74030756 0.6936243  0.78646617 0.53556619 0.53429783
  0.64074365 0.17738791 0.20692758 0.26889474 0.57278835 0.5168169
  0.44692398 0.69527363 0.31634349 0.34422263 0.57794787 0.46910755
  0.54992622 0.55563735 0.31708902 0.40271132 0.3264411  0.42580409
  0.50841389 0.52317449]]
[0 2 2 0 0 1 0 1 2 0 0 2 0 1 1 0 0 0 0 2 0 0 2 1 1 2 1 0 1 2 1 1 0 0 0 1 1
 1 1 1 0 0 1 0 0 1 0 0 0 1 0 1 1 0

In [17]:
#Create a visualization of the results with 2 or 3 variables that you think will best
#differentiate the clusters
fig = px.scatter_3d(nba_data, x="PTS", y="eFG%", z="MP", color=kmeans_nba.labels_,
                    title="Aye vs. Nay vs. Other votes for Democrat-introduced bills")
fig.write_html("my_plot.html")

In [20]:
#Evaluate the quality of the clustering using total variance explained and silhouette scores

624.4258264368938
391.03617219006213
0.6262331819639771


In [ ]:
# Variance explained:

total_sum_squares = np.sum((x - np.mean(x))**2)
total = np.sum(total_sum_squares)
print(total)

between_SSE = (total-kmeans_nba.inertia_)
print(between_SSE)

Var_explained = between_SSE/total
print(Var_explained)

In [25]:
# Silhouette Method
from sklearn.metrics import silhouette_score

# Run NbClust
silhouette_scores = []
for k in range(2, 11):
    kmeans_obj = KMeans(n_clusters=k, algorithm="lloyd", random_state=1).fit(x)
    silhouette_scores.append(silhouette_score(x, kmeans_obj.labels_))

best_nc = silhouette_scores.index(max(silhouette_scores))+2
silhouette_scores

[np.float64(0.33215373902332224),
 np.float64(0.18294607873583396),
 np.float64(0.18616308944964072),
 np.float64(0.16502126632302253),
 np.float64(0.160126184319006),
 np.float64(0.15099448005802024),
 np.float64(0.12798489993350362),
 np.float64(0.1288898718542961),
 np.float64(0.12900945811719017)]

In [28]:
#Determine the ideal number of clusters using the elbow method and the silhouette coefficient

wcss = []
for i in range(1, 11):
    kmeans_obj_nba = KMeans(n_clusters=i, random_state=1).fit(x)
    wcss.append(kmeans_obj_nba.inertia_)

elbow_data_nba = pd.DataFrame({"k": range(1, 11), "wcss": wcss})

In [29]:
#Visualize the results of the elbow method
fig = px.line(elbow_data_nba, x="k", y="wcss", title="Elbow Method for Optimal k")
fig.update_layout(xaxis_title="Number of Clusters (k)", yaxis_title="Within-Cluster Sum of Squares (WCSS)")
fig.write_html("my_plot_2.html")

In [31]:
#Use the recommended number of cluster (assuming it's different) to retrain your model and visualize the results
kmeans_nba_2 = KMeans(n_clusters=2, random_state=67).fit(x)
print(kmeans_nba_2.cluster_centers_) 
print(kmeans_nba_2.labels_)
print(kmeans_nba_2.inertia_)

[[0.30382775 0.5579922  0.14626123 0.43975564 0.20427632 0.24081754
  0.484375   0.18603801 0.20583108 0.31824671 0.18204087 0.17545071
  0.37380702 0.61486768 0.09397507 0.09706783 0.62744615 0.18921625
  0.20533079 0.20739787 0.1387366  0.26958732 0.1368656  0.17516447
  0.31162728 0.20691456]
 [0.33778966 0.72847575 0.66818109 0.82737628 0.52917409 0.6045287
  0.49481758 0.41931736 0.44578515 0.346      0.45000695 0.44433611
  0.36408715 0.62734124 0.28957688 0.29017354 0.69345719 0.24211424
  0.39142386 0.36395631 0.38259501 0.4560309  0.20074697 0.43096405
  0.44871282 0.54650238]]
[0 1 1 0 0 1 0 1 1 0 0 1 0 1 1 0 0 0 0 1 0 0 0 1 1 1 1 0 1 1 1 1 0 0 0 1 1
 0 1 1 0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 1 0 0 1 0 0 1 0 1 1 0 1 0 0 0 1 0 0 0
 0 0 1 0 1 0 1 0 0 0 0 0 0 0 1 1 1 1 1 1 1 0 0 1 0 0 1 1 0 0 0 0 1 0 0 0 0
 1 1 1 0 1 0 0 1 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 1
 1 0 1 1 0 0 0 0 0 0 0 1 1 1 0 1 1 1 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 1 1 1
 1 0 0 0 0 0 1 1 0 0 0 1 0 0 0 

In [34]:
#Once again evaluate the quality of the clustering using total variance explained and silhouette scores
silhouette_scores_2 = []
kmeans_obj = KMeans(n_clusters=2, algorithm="lloyd", random_state=1).fit(x)
    silhouette_scores.append(silhouette_score(x, kmeans_obj.labels_))

best_nc = silhouette_scores.index(max(silhouette_scores))+2
silhouette_scores_2

KMeans(n_clusters=2, random_state=1)

In [ ]:
wcss = []
for i in range(1, 11):
    kmeans_obj_nba = KMeans(n_clusters=i, random_state=1).fit(x)
    wcss.append(kmeans_obj_nba.inertia_)

elbow_data_nba = pd.DataFrame({"k": range(1, 11), "wcss": wcss})

In [35]:
#Use the model to select players for Mr. Rooney to consider
# Add cluster labels to your dataframe
nba_data['Cluster'] = kmeans_nba_2.labels_

# Calculate performance metrics (example)
nba_data['Performance_Score'] = (nba_data['PTS'] + nba_data['MP'] + nba_data['eFG%'])/3

# Find undervalued players
undervalued_players = nba_data[
    (nba_data['Performance_Score'] > nba_data['Performance_Score'].mean()) & 
    (nba_data['Salary'] < nba_data['Salary'].mean())
]

print("\nPotential Target Players:")
print(undervalued_players[['Player', 'PTS', 'MP', 'eFG%', 'Salary']].head())


Potential Target Players:
Empty DataFrame
Columns: [Player, PTS, MP, eFG%, Salary]
Index: []


#Write up the results in a separate notebook with supporting visualizations and 
an overview of how and why you made the choices you did. This should be at least 
500 words and should be written for a non-technical audience.